In [ ]:
pip install openpyxl

In [ ]:
pip install nbformat

In [ ]:
pip install plotly


In [9]:
GENERATE_3D_GIF = True   # True = create rotating GIF, False = skip

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import plotly.graph_objects as go

# -------------------------------
# Smart File Loader (CSV + Excel + ODS)
# -------------------------------
def load_well_dataframe(file_path):
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(file_path)

    elif suffix in [".xls", ".xlsx", ".xlsm", ".xlsb"]:
        df = pd.read_excel(file_path)

    elif suffix == ".ods":
        df = pd.read_excel(file_path, engine="odf")

    else:
        raise ValueError("Unsupported file format")

    return df


# -------------------------------
# Find required columns
# -------------------------------
def find_column(df, possible_names):
    for col in df.columns:
        if col.strip().lower() in possible_names:
            return col
    return None


# -------------------------------
# Compute trajectory (Minimum Curvature)
# -------------------------------
def compute_well_trajectory(df):
    md_col  = find_column(df, {"measured_depth","md","dept","depth"})
    inc_col = find_column(df, {"inclination","inc","incl"})
    azi_col = find_column(df, {"azimuth","azi","azim"})

    if not (md_col and inc_col and azi_col):
        raise ValueError("Required columns (Measured Depth, Inclination, Azimuth) not found")

    md  = df[md_col].astype(float).values
    inc = np.radians(df[inc_col].astype(float).values)
    azi = np.radians(df[azi_col].astype(float).values)

    n = len(md)

    X = np.zeros(n)
    Y = np.zeros(n)
    Z = np.zeros(n)

    for i in range(1, n):
        dmd = md[i] - md[i-1]

        inc1, inc2 = inc[i-1], inc[i]
        azi1, azi2 = azi[i-1], azi[i]

        dogleg = np.arccos(
            np.cos(inc2-inc1) -
            np.sin(inc1)*np.sin(inc2)*(1-np.cos(azi2-azi1))
        )

        rf = 1 if dogleg == 0 else 2/dogleg * np.tan(dogleg/2)

        dX = dmd/2 * (np.sin(inc1)*np.sin(azi1) + np.sin(inc2)*np.sin(azi2)) * rf
        dY = dmd/2 * (np.sin(inc1)*np.cos(azi1) + np.sin(inc2)*np.cos(azi2)) * rf
        dZ = dmd/2 * (np.cos(inc1) + np.cos(inc2)) * rf

        X[i] = X[i-1] + dX
        Y[i] = Y[i-1] + dY
        Z[i] = Z[i-1] + dZ

    return md, X, Y, Z


# -------------------------------
# Plot and export 3 views in one image
# -------------------------------
from io import BytesIO

from io import BytesIO

def plot_and_export_views(md, X, Y, Z, output_path, output_3d_path):
    
    images = []

    # --- Top View ---
    buf = BytesIO()
    plt.figure()
    plt.plot(X, Y)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.title("Top View")
    plt.xlabel("X"); plt.ylabel("Y")
    plt.savefig(buf, format='png', dpi=300, bbox_inches="tight")
    plt.close()
    buf.seek(0)
    images.append(Image.open(buf))

    # --- Cross Section (Depth Downwards) ---
    buf = BytesIO()
    plt.figure()
    plt.plot(X, Z)
    plt.gca().invert_yaxis()
    plt.title("Cross Section View")
    plt.xlabel("X"); plt.ylabel("TVD")
    plt.savefig(buf, format='png', dpi=300, bbox_inches="tight")
    plt.close()
    buf.seek(0)
    images.append(Image.open(buf))

    # --- Combine Top + Cross ---
    widths, heights = zip(*(i.size for i in images))
    combined = Image.new("RGB", (sum(widths), max(heights)))

    x_offset = 0
    for img in images:
        combined.paste(img, (x_offset, 0))
        x_offset += img.size[0]

    combined.save(output_path)
    print(f"Saved combined 2-view image → {output_path}")

    # --- Static 3D Plot ---
    from mpl_toolkits.mplot3d import Axes3D

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    ax.plot(X, Y, -Z, color='blue')
    ax.scatter(X[0], Y[0], -Z[0], color='green', s=60)
    ax.quiver(X[-1], Y[-1], -Z[-1], 
          X[-1]-X[-2], Y[-1]-Y[-2], (-Z[-1])-(-Z[-2]),
          length=1.5, color='red', arrow_length_ratio=0.3)

    ax.set_title("3D Well Trajectory")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("TVD")
    plt.savefig(output_3d_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved static 3D image → {output_3d_path}")

    # --- Rotating 3D GIF (Optional) ---
    if GENERATE_3D_GIF:
        import matplotlib.animation as animation

        gif_path = output_3d_path.with_suffix(".gif")

        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        # Main trajectory line
        ax.plot(X, Y, -Z, color='blue')

        # Well head (start point) → green dot
        ax.scatter(X[0], Y[0], -Z[0], color='green', s=60)

        # Well end (last point) → red arrow
        ax.quiver(X[-1], Y[-1], -Z[-1], 
          X[-1]-X[-2], Y[-1]-Y[-2], (-Z[-1])-(-Z[-2]),
          length=1.5, color='red', arrow_length_ratio=0.3)


        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("TVD")
        ax.set_title("3D Well Trajectory – 360° View")

        # Keep vertical (TVD) axis as rotation axis
        def rotate(angle):
            ax.view_init(elev=20, azim=angle)
            return fig,

        #Animation Controls
        ani = animation.FuncAnimation(
            fig, rotate,
            frames=np.arange(0, 360, 3),
            interval=50
        )

        ani.save(gif_path, writer='pillow', dpi=120)
        plt.close()

        print(f"Saved rotating 3D GIF → {gif_path}")


# -------------------------------
# Interactive 3D in Jupyter
# -------------------------------
def show_interactive_3d(X, Y, Z):
    import plotly.graph_objects as go
    import plotly.io as pio
    
    # Force browser rendering instead of Jupyter MIME
    pio.renderers.default = "browser"

    fig = go.Figure(data=[go.Scatter3d(
        x=X, y=Y, z=-Z,
        mode='lines'
    )])

    fig.update_layout(
        title="Interactive 3D Well Trajectory",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="TVD"
        )
    )

    fig.show()


# -------------------------------
# MASTER FUNCTION
# -------------------------------
def generate_well_plots_from_file(file_path):
    file_path = Path(file_path)

    df = load_well_dataframe(file_path)
    md, X, Y, Z = compute_well_trajectory(df)

    output_dir = file_path.parent / "well_trajectory"
    output_dir.mkdir(exist_ok=True)

    output_2view = output_dir / f"{file_path.stem}_well_views.png"
    output_3d    = output_dir / f"{file_path.stem}_well_3D.png"

    plot_and_export_views(md, X, Y, Z, output_2view, output_3d)
    show_interactive_3d(X, Y, Z)


In [13]:
generate_well_plots_from_file("/Users/apple/Downloads/welldata/2011/SCHOONEBEEK-2302/structured/NLOG_GS_PUB_SCH-2302.xlsx")


Saved combined 2-view image → /Users/apple/Downloads/welldata/2011/SCHOONEBEEK-2302/structured/well_trajectory/NLOG_GS_PUB_SCH-2302_well_views.png
Saved static 3D image → /Users/apple/Downloads/welldata/2011/SCHOONEBEEK-2302/structured/well_trajectory/NLOG_GS_PUB_SCH-2302_well_3D.png
Saved rotating 3D GIF → /Users/apple/Downloads/welldata/2011/SCHOONEBEEK-2302/structured/well_trajectory/NLOG_GS_PUB_SCH-2302_well_3D.gif
